In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # repo root, for `kernels` / `model`

import torch
import torch.nn.functional as F
from triton.testing import do_bench

from model.llama3_1b import Llama
from kernels import fa2_fwd, flash_decode

print(torch.cuda.get_device_name(), torch.__version__)

MAX_SEQ = 131072 + 64
model, cfg = Llama.from_hf(max_seq_len=MAX_SEQ)
model.setup_caches(1, MAX_SEQ, torch.float16, "cuda")
print(f"cache: {model.cache_bytes()/1e9:.2f} GB")

IMPLS = ["sdpa", "sdpa_gqa", "triton"]

NVIDIA A100-SXM4-40GB 2.8.0+cu128


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

cache: 4.30 GB


In [2]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("NousResearch/Llama-3.2-1B")
ids = tok("The key idea behind FlashAttention is", return_tensors="pt").input_ids.cuda()

@torch.inference_mode()
def generate(impl, n_new=64):
    model.set_attn_impl(impl)
    logits = model(ids, start_pos=0, use_cache=True, last_only=True)   # prefill
    first_logits = logits[0, -1].float()
    next_tok = logits[:, -1].argmax(-1, keepdim=True)
    tokens = [next_tok.item()]
    pos = ids.shape[1]
    for _ in range(n_new - 1):                                         # decode
        logits = model(next_tok, start_pos=pos, use_cache=True)
        next_tok = logits[:, -1].argmax(-1, keepdim=True)
        tokens.append(next_tok.item())
        pos += 1
    return first_logits, tokens

ref_logits, ref_tokens = generate("sdpa")
for impl in ["sdpa_gqa", "triton"]:
    logits, tokens = generate(impl)
    diff = (logits - ref_logits).abs().max().item()
    same = sum(a == b for a, b in zip(tokens, ref_tokens))
    print(f"{impl:9s}  max logit diff = {diff:.3e}   tokens matching = {same}/64")

print(tok.decode(tokens))

sdpa_gqa   max logit diff = 0.000e+00   tokens matching = 64/64


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


triton     max logit diff = 1.562e-02   tokens matching = 64/64
 that the attentional system is a dynamic system that is constantly changing. This means that the attentional system is not static, but rather it is constantly changing and adapting to the environment. This is why it is important to understand the dynamics of the attentional system and how it changes over time.
The key idea behind Flash


In [3]:
def bench_decode(S, B=1, num_splits=None):
    k = torch.randn(B, 8, S, 64, device="cuda", dtype=torch.float16)
    v = torch.randn(B, 8, S, 64, device="cuda", dtype=torch.float16)
    q = torch.randn(B, 32, 64, device="cuda", dtype=torch.float16)
    seq_lens = torch.full((B,), S, device="cuda", dtype=torch.int32)
    kv_bytes = 2 * k.numel() * 2

    times = {
        "read only":  do_bench(lambda: (k.sum(), v.sum())),
        "fa2 L3":     do_bench(lambda: fa2_fwd(q[:, :, None], k, v, causal=False)),
        "fd 1 split": do_bench(lambda: flash_decode(q, k, v, seq_lens, num_splits=1)),
        "fd auto":    do_bench(lambda: flash_decode(q, k, v, seq_lens, num_splits=num_splits)),
        "sdpa gqa":   do_bench(lambda: F.scaled_dot_product_attention(
                                   q[:, :, None], k, v, enable_gqa=True)),
    }
    line = f"B={B:2d} S={S:6d} | "
    for name, ms in times.items():
        gbps = kv_bytes / (ms * 1e-3) / 1e9
        line += f"{name}: {ms*1e3:7.1f}us {gbps:5.0f}GB/s | "
    print(line)

for S in [512, 1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072]:
    bench_decode(S)

print("\n batch check")
bench_decode(32768, B=16)

print("\n split sweep, S=32k")
for ns in [1, 2, 4, 8, 16, 32, 64]:
    print("splits =", ns)
    bench_decode(32768, num_splits=ns)

B= 1 S=   512 | read only:    31.6us    33GB/s | fa2 L3:    27.6us    38GB/s | fd 1 split:    21.2us    49GB/s | fd auto:    13.8us    76GB/s | sdpa gqa:    20.3us    52GB/s | 
B= 1 S=  1024 | read only:    27.4us    77GB/s | fa2 L3:    37.8us    55GB/s | fd 1 split:    28.0us    75GB/s | fd auto:    16.3us   129GB/s | sdpa gqa:    21.6us    97GB/s | 
B= 1 S=  2048 | read only:    28.5us   147GB/s | fa2 L3:    66.8us    63GB/s | fd 1 split:    44.6us    94GB/s | fd auto:    18.3us   229GB/s | sdpa gqa:    23.8us   176GB/s | 
B= 1 S=  4096 | read only:    32.8us   256GB/s | fa2 L3:   124.3us    68GB/s | fd 1 split:    77.9us   108GB/s | fd auto:    22.6us   371GB/s | sdpa gqa:    31.5us   266GB/s | 
B= 1 S=  8192 | read only:    43.5us   385GB/s | fa2 L3:   242.5us    69GB/s | fd 1 split:   144.6us   116GB/s | fd auto:    32.9us   510GB/s | sdpa gqa:    38.9us   431GB/s | 
B= 1 S= 16384 | read only:    62.8us   534GB/s | fa2 L3:   477.0us    70GB/s | fd 1 split:   278.0us   121GB/s | fd

In [4]:
@torch.inference_mode()
def ttft(impl, P):
    model.set_attn_impl(impl)
    prompt = torch.randint(0, cfg.vocab_size, (1, P), device="cuda")
    return do_bench(lambda: model(prompt, start_pos=0, use_cache=True, last_only=True))

for P in [512, 2048, 8192]:
    print(f"prompt {P:5d}: " + "  ".join(f"{i} {ttft(i, P):.2f}ms" for i in IMPLS))

prompt   512: sdpa 12.93ms  sdpa_gqa 11.18ms  triton 13.14ms
prompt  2048: sdpa 28.57ms  sdpa_gqa 28.00ms  triton 28.59ms
prompt  8192: sdpa 129.86ms  sdpa_gqa 128.89ms  triton 138.09ms


In [5]:
@torch.inference_mode()
def tpot(impl, ctx):
    model.set_attn_impl(impl)
    tok1 = torch.randint(0, cfg.vocab_size, (1, 1), device="cuda")
    return do_bench(lambda: model(tok1, start_pos=ctx - 1, use_cache=True))

for ctx in [1024, 8192, 32768, 131072]:
    print(f"context {ctx:6d}: " + "  ".join(f"{i} {tpot(i, ctx):.3f}ms" for i in IMPLS))

context   1024: sdpa 11.791ms  sdpa_gqa 10.820ms  triton 13.332ms
context   8192: sdpa 11.650ms  sdpa_gqa 10.880ms  triton 13.395ms
context  32768: sdpa 15.356ms  sdpa_gqa 11.027ms  triton 13.551ms
context 131072: sdpa 53.794ms  sdpa_gqa 10.959ms  triton 13.400ms


In [6]:
from torch.profiler import profile, ProfilerActivity

@torch.inference_mode()
def overhead(impl, ctx=8192, n=20):
    model.set_attn_impl(impl)
    tok1 = torch.randint(0, cfg.vocab_size, (1, 1), device="cuda")
    step = lambda: model(tok1, start_pos=ctx - 1, use_cache=True)

    wall_ms = do_bench(step)                                   # includes idle gaps

    with profile(activities=[ProfilerActivity.CUDA]) as prof:  # GPU busy time only
        for _ in range(n):
            step()
        torch.cuda.synchronize()
    gpu_ms = sum(e.self_device_time_total for e in prof.key_averages()) / n / 1e3

    print(f"{impl:9s} wall {wall_ms:.3f}ms   gpu busy {gpu_ms:.3f}ms   gap {wall_ms - gpu_ms:.3f}ms")

for impl in IMPLS:
    overhead(impl)

sdpa      wall 11.669ms   gpu busy 7.392ms   gap 4.276ms
sdpa_gqa  wall 13.441ms   gpu busy 5.143ms   gap 8.298ms
triton    wall 16.223ms   gpu busy 5.000ms   gap 11.222ms
